# 04 · Fine-tune a small LLM with QLoRA on a free T4

**Runtime → T4 GPU**, then run all (~30 min). Base model → eval → QLoRA fine-tune → eval again → before/after table.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/PRIBINKBENNY/04-finetune.git project
%cd project
!pip install -q -r requirements.txt

In [ ]:
!python -m finetune.data --train 2000 --test 200

## Before: base model

In [ ]:
!python -m finetune.evaluate --backend hf --model Qwen/Qwen2.5-0.5B-Instruct --tag base --limit 200

## Fine-tune (QLoRA, 4-bit base + LoRA r=16)

In [ ]:
!python -m finetune.train --config configs/qlora_t4.yaml

## After: base + adapter

In [ ]:
!python -m finetune.evaluate --backend hf --model Qwen/Qwen2.5-0.5B-Instruct --adapter outputs/qwen0.5b-jobs-lora/adapter --tag finetuned --limit 200

In [ ]:
!python -m finetune.evaluate --compare results/base.json results/finetuned.json --readme README.md

In [ ]:
import json; d=json.load(open('results/finetuned.json'))['details']
for r in d[:3]: print(r['text'], '\n->', r['pred'], '\n')

## Save
Download the results and adapter (a few MB). Commit `results/*.json` + README; optionally `python -m finetune.merge ... --push <hf-user>/qwen0.5b-job-extractor`.

In [ ]:
!zip -qr finetune_results.zip results README.md outputs/qwen0.5b-jobs-lora/adapter outputs/qwen0.5b-jobs-lora/train_log.json
from google.colab import files; files.download('finetune_results.zip')